In [1]:
import csv
from pathlib import Path
from datetime import datetime, timedelta

In [2]:
file_path = Path("../data/raw/india_daily_weather_2000_2024_raw.csv")

In [3]:
try:
    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        print("CSV file opened successfully!")
        print("Column names:", reader.fieldnames)

except FileNotFoundError:
    print("Error: Dataset file not found.")

except Exception as e:
    print("Error:", e)


CSV file opened successfully!
Column names: ['city', 'date', 'temp_max', 'temp_min', 'wind_speed_kmh', 'pressure_hpa', 'sunshine_hours', 'rainfall_mm', 'apparent_temperature_max', 'apparent_temperature_min', 'weather_code', 'wind_gusts_10m_max', 'wind_direction_10m_dominant']


In [4]:
try:
    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            print(row)
            break

except FileNotFoundError:
    print("Error: Dataset file not found.")

except Exception as e:
    print("Error:", e)


{'city': 'Ahmedabad', 'date': '1/1/2000', 'temp_max': '29.9', 'temp_min': '15.3', 'wind_speed_kmh': '14.5', 'pressure_hpa': '1007.21', 'sunshine_hours': '10.28', 'rainfall_mm': '0', 'apparent_temperature_max': '29.1', 'apparent_temperature_min': '12.8', 'weather_code': '0', 'wind_gusts_10m_max': '30.2', 'wind_direction_10m_dominant': '60'}


In [5]:
try:
    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        row = next(reader)
        temp_max = float(row["temp_max"])
        temp_min = float(row["temp_min"])
        wind_speed = float(row["wind_speed_kmh"])
        pressure = float(row["pressure_hpa"])

        if not (-5 <= temp_max <= 55):
            raise ValueError("Maximum temperature out of range")

        if not (-5 <= temp_min <= 55):
            raise ValueError("Minimum temperature out of range")

        if wind_speed < 0:
            raise ValueError("Negative wind speed")

        if not (900 <= pressure <= 1050):
            raise ValueError("Pressure out of range")

        print("First record is valid!")

except FileNotFoundError:
    print("Error: Dataset file not found.")

except ValueError as e:
    print("Invalid record:", e)


First record is valid!


In [32]:
valid_count = 0
invalid_count = 0
error_log = []

try:
    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            try:
                temp_max = float(row["temp_max"])
                temp_min = float(row["temp_min"])
                wind_speed = float(row["wind_speed_kmh"])
                pressure = float(row["pressure_hpa"])

                if not (-5 <= temp_max <= 55):
                    raise ValueError("Maximum temperature out of range")

                if not (-5 <= temp_min <= 55):
                    raise ValueError("Minimum temperature out of range")

                if wind_speed < 0:
                    raise ValueError("Negative wind speed")

                if not (900 <= pressure <= 1050):
                    raise ValueError("Pressure out of range")

                valid_count += 1

            except ValueError as e:
                invalid_count += 1
                error_log.append({
                    "city": row["city"],
                    "date": row["date"],
                    "error": str(e)
                })

    print("Valid records:", valid_count)
    print("Invalid records:", invalid_count)

except FileNotFoundError:
    print("Error: Dataset file not found.")

except Exception as e:
    print("Error:", e)


Valid records: 91320
Invalid records: 0


In [33]:
city_dates = {}

try:
    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            city = row["city"]
            date = datetime.strptime(row["date"], "%m/%d/%Y").date()

            if city not in city_dates:
                city_dates[city] = set()

            city_dates[city].add(date)

    print("Cities found:", len(city_dates))
    print("Date range:", min(min(dates) for dates in city_dates.values()),
          "to", max(max(dates) for dates in city_dates.values()))

except FileNotFoundError:
    print("Error: Dataset file not found.")

except ValueError as e:
    print("Date conversion error:", e)


Cities found: 10
Date range: 2000-01-01 to 2024-12-31


In [34]:
start_date = datetime(2000, 1, 1).date()
end_date = datetime(2024, 12, 31).date()
expected_dates = set()
current_date = start_date

while current_date <= end_date:
    expected_dates.add(current_date)
    current_date += timedelta(days=1)

for city, dates in city_dates.items():
    missing_dates = expected_dates - dates

    print(city, ":", len(missing_dates), "missing days")

Ahmedabad : 0 missing days
Bangalore : 0 missing days
Chennai : 0 missing days
Delhi : 0 missing days
Hyderabad : 0 missing days
Jaipur : 0 missing days
Kolkata : 0 missing days
Lucknow : 0 missing days
Mumbai : 0 missing days
Pune : 0 missing days


In [35]:
total_expected = len(expected_dates) * len(city_dates)
total_actual = sum(len(dates) for dates in city_dates.values())

print("Expected records:", total_expected)
print("Actual records:", total_actual)
print("Missing records:", total_expected - total_actual)

Expected records: 91320
Actual records: 91320
Missing records: 0


In [10]:
log_path = Path("../data/processed/q1_error_log.txt")

try:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "w", encoding="utf-8") as log_file:
        if error_log:
            for error in error_log:
                log_file.write(
                    f"City: {error['city']}, "
                    f"Date: {error['date']}, "
                    f"Error: {error['error']}\n"
                )
        else:
            log_file.write("No invalid records.\n")
    print("Error log created successfully!")
    print("Location:", log_path)

except OSError as e:
    print("File writing error:", e)

Error log created successfully!
Location: ..\data\processed\q1_error_log.txt


In [11]:
with open(log_path, "r", encoding="utf-8") as log_file:
    print(log_file.read())

No invalid records.



In [12]:
output_dir = Path("../data/processed")
clean_file = output_dir / "clean_india_weather_25yr.csv"
audit_file = output_dir / "wind_solar_audit.log"

output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory ready:", output_dir)
print("Clean CSV:", clean_file.name)
print("Audit log:", audit_file.name)

Output directory ready: ..\data\processed
Clean CSV: clean_india_weather_25yr.csv
Audit log: wind_solar_audit.log


In [13]:
clean_records = []

try:
    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            try:
                temp_max = float(row["temp_max"])
                temp_min = float(row["temp_min"])
                wind_speed = float(row["wind_speed_kmh"])
                pressure = float(row["pressure_hpa"])

                if (-5 <= temp_max <= 55 and
                    -5 <= temp_min <= 55 and
                    wind_speed >= 0 and
                    900 <= pressure <= 1050):
                    clean_records.append(row)

            except ValueError:
                continue
    print("Sanitized records collected:", len(clean_records))

except FileNotFoundError:
    print("Error: Dataset file not found.")
except OSError as e:
    print("File reading error:", e)

Sanitized records collected: 91320


In [14]:
try:
    with open(clean_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=clean_records[0].keys())
        writer.writeheader()
        writer.writerows(clean_records)

    print("Clean CSV exported successfully!")
    print("Records written:", len(clean_records))

except OSError as e:
    print("File writing error:", e)

Clean CSV exported successfully!
Records written: 91320


In [15]:
try:
    with open(clean_file, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        exported_count = sum(1 for row in reader)

    print("File exists:", clean_file.exists())
    print("Exported records:", exported_count)
    print("Verification:", "Passed" if exported_count == len(clean_records) else "Failed")

except OSError as e:
    print("File verification error:", e)

File exists: True
Exported records: 91320
Verification: Passed


In [16]:
missing_data = {}

try:
    with open(clean_file, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            city = row["city"]
            if city not in missing_data:
                missing_data[city] = {
                    "sunshine": 0,
                    "wind": 0
                }

            if not row["sunshine_hours"].strip():
                missing_data[city]["sunshine"] += 1

            if not row["wind_speed_kmh"].strip():
                missing_data[city]["wind"] += 1

    for city, counts in missing_data.items():
        print(
            city,
            "| Missing sunshine:", counts["sunshine"],
            "| Missing wind:", counts["wind"]
        )

except OSError as e:
    print("File reading error:", e)

Ahmedabad | Missing sunshine: 0 | Missing wind: 0
Bangalore | Missing sunshine: 0 | Missing wind: 0
Chennai | Missing sunshine: 0 | Missing wind: 0
Delhi | Missing sunshine: 0 | Missing wind: 0
Hyderabad | Missing sunshine: 0 | Missing wind: 0
Jaipur | Missing sunshine: 0 | Missing wind: 0
Kolkata | Missing sunshine: 0 | Missing wind: 0
Lucknow | Missing sunshine: 0 | Missing wind: 0
Mumbai | Missing sunshine: 0 | Missing wind: 0
Pune | Missing sunshine: 0 | Missing wind: 0


In [17]:
try:
    with open(audit_file, "w", encoding="utf-8") as file:
        file.write("WIND AND SOLAR DATA AUDIT REPORT\n")
        file.write("=" * 40 + "\n")

        for city, counts in missing_data.items():
            file.write(
                f"City: {city} | "
                f"Missing Sunshine: {counts['sunshine']} | "
                f"Missing Wind: {counts['wind']}\n"
            )

        total_gaps = sum(
            counts["sunshine"] + counts["wind"]
            for counts in missing_data.values()
        )

        if total_gaps == 0:
            file.write("\nNo missing sunshine or wind gaps found.\n")

    print("Audit log created successfully!")
    print("Location:", audit_file)

except OSError as e:
    print("Audit log writing error:", e)

Audit log created successfully!
Location: ..\data\processed\wind_solar_audit.log


In [18]:
try:
    with open(audit_file, "r", encoding="utf-8") as file:
        print(file.read())

except FileNotFoundError:
    print("Audit log not found.")

except OSError as e:
    print("File reading error:", e)

WIND AND SOLAR DATA AUDIT REPORT
City: Ahmedabad | Missing Sunshine: 0 | Missing Wind: 0
City: Bangalore | Missing Sunshine: 0 | Missing Wind: 0
City: Chennai | Missing Sunshine: 0 | Missing Wind: 0
City: Delhi | Missing Sunshine: 0 | Missing Wind: 0
City: Hyderabad | Missing Sunshine: 0 | Missing Wind: 0
City: Jaipur | Missing Sunshine: 0 | Missing Wind: 0
City: Kolkata | Missing Sunshine: 0 | Missing Wind: 0
City: Lucknow | Missing Sunshine: 0 | Missing Wind: 0
City: Mumbai | Missing Sunshine: 0 | Missing Wind: 0
City: Pune | Missing Sunshine: 0 | Missing Wind: 0

No missing sunshine or wind gaps found.



### Q 3

In [19]:
class WindSolarResourceEstimator:

    def __init__(self):
        self.data = None
        self.hub_height = 100.0


estimator = WindSolarResourceEstimator()

print("Data:", estimator.data)
print("Hub height:", estimator.hub_height)

Data: None
Hub height: 100.0


In [20]:
class WindSolarResourceEstimator:

    def __init__(self):
        self.data = None
        self.hub_height = 100.0

    def load_from_csv(self, file_path):
        try:
            with open(file_path, "r", newline="", encoding="utf-8") as file:
                reader = csv.DictReader(file)
                self.data = list(reader)

            print("Dataset loaded successfully!")
            print("Total records:", len(self.data))

        except FileNotFoundError:
            print("Error: CSV file not found.")

    def compute_power_density(self, hub_height=100.0):

        if hub_height <= 0:
            raise ValueError("Hub height must be greater than zero.")
        if self.data is None:
            raise ValueError("Load the dataset first.")

        self.hub_height = hub_height

        for row in self.data:
            temperature = (float(row["temp_max"]) + float(row["temp_min"])) / 2
            temp_kelvin = temperature + 273.15
            pressure_pa = float(row["pressure_hpa"]) * 100
            wind_speed = float(row["wind_speed_kmh"]) / 3.6
            air_density = pressure_pa / (287.05 * temp_kelvin)
            power_density = 0.5 * air_density * (wind_speed ** 3)
            row["air_density_rho"] = air_density
            row["wind_power_density"] = power_density

        print("Wind power density calculated successfully!")


    def summary_report(self):

        if self.data is None:
            raise ValueError("Load the dataset first.")
        total_records = len(self.data)

        average_density = sum(row["wind_power_density"] for row in self.data) / total_records

        print("WIND AND SOLAR RESOURCE SUMMARY")
        print("--------------------------------")
        print("Total records:", total_records)
        print("Hub height:", self.hub_height, "metres")
        print("Average wind power density:",
              round(average_density, 2), "W/m²")

In [21]:
estimator = WindSolarResourceEstimator()
estimator.load_from_csv(clean_file)

print("First record:", estimator.data[0])

Dataset loaded successfully!
Total records: 91320
First record: {'city': 'Ahmedabad', 'date': '1/1/2000', 'temp_max': '29.9', 'temp_min': '15.3', 'wind_speed_kmh': '14.5', 'pressure_hpa': '1007.21', 'sunshine_hours': '10.28', 'rainfall_mm': '0', 'apparent_temperature_max': '29.1', 'apparent_temperature_min': '12.8', 'weather_code': '0', 'wind_gusts_10m_max': '30.2', 'wind_direction_10m_dominant': '60'}


In [22]:
estimator.compute_power_density()

print("Air density:", estimator.data[0]["air_density_rho"])
print("Wind power density:", estimator.data[0]["wind_power_density"])

Wind power density calculated successfully!
Air density: 1.186417992924498
Wind power density: 38.76182649262096


In [23]:
estimator.summary_report()

WIND AND SOLAR RESOURCE SUMMARY
--------------------------------
Total records: 91320
Hub height: 100.0 metres
Average wind power density: 62.04 W/m²


In [24]:
try:
    estimator.compute_power_density(0)
except ValueError as e:
    print("ValueError caught:", e)

ValueError caught: Hub height must be greater than zero.


In [25]:
import sys
sys.path.append("..")

from src.ingestion import validate_record

test_row = {
    "city": "Kochi",
    "date": "2000-01-01",
    "temp_max": "30",
    "temp_min": "22",
    "wind_speed_kmh": "15",
    "pressure_hpa": "1005"
}

print(validate_record(test_row))

(True, 'Valid record')


In [26]:
import importlib
import src.ingestion as ingestion

importlib.reload(ingestion)

valid_records, error_log = ingestion.load_and_validate_csv(
    "../data/raw/india_daily_weather_2000_2024_raw.csv"
)

print("Valid records:", len(valid_records))
print("Invalid records:", len(error_log))

Valid records: 91320
Invalid records: 0


In [27]:
import importlib
import src.ingestion as ingestion

importlib.reload(ingestion)

clean_file = ingestion.export_clean_data(
    valid_records,
    "../data/processed"
)

print("Saved file:", clean_file)

Clean CSV exported successfully!
Records written: 91320
Saved file: ..\data\processed\clean_india_weather_25yr.csv


In [28]:
import importlib
import src.ingestion as ingestion

importlib.reload(ingestion)

audit_file = ingestion.create_audit_log(
    valid_records,
    "../data/processed"
)

print("Audit file:", audit_file)

Audit log created successfully!
Audit file: ..\data\processed\wind_solar_audit.log


In [29]:
with open(audit_file, "r", encoding="utf-8") as file:
    print(file.read())
    

WIND AND SOLAR DATA AUDIT REPORT
City: Ahmedabad | Missing Sunshine: 0 | Missing Wind: 0
City: Bangalore | Missing Sunshine: 0 | Missing Wind: 0
City: Chennai | Missing Sunshine: 0 | Missing Wind: 0
City: Delhi | Missing Sunshine: 0 | Missing Wind: 0
City: Hyderabad | Missing Sunshine: 0 | Missing Wind: 0
City: Jaipur | Missing Sunshine: 0 | Missing Wind: 0
City: Kolkata | Missing Sunshine: 0 | Missing Wind: 0
City: Lucknow | Missing Sunshine: 0 | Missing Wind: 0
City: Mumbai | Missing Sunshine: 0 | Missing Wind: 0
City: Pune | Missing Sunshine: 0 | Missing Wind: 0

Audit completed successfully.



In [30]:
import importlib
import src.ingestion as ingestion

importlib.reload(ingestion)

estimator = ingestion.WindSolarResourceEstimator()

estimator.load_from_csv(
    "../data/processed/clean_india_weather_25yr.csv"
)

estimator.compute_power_density()

estimator.summary_report()

Dataset loaded successfully!
Total records: 91320
Wind power density calculated successfully!

WIND AND SOLAR RESOURCE SUMMARY
--------------------------------
Total records: 91320
Hub height: 100.0 metres
Average wind power density: 62.04 W/m²


In [31]:
try:
    estimator.compute_power_density(hub_height=0)

except ValueError as e:
    print("ValueError caught:", e)

ValueError caught: Hub height must be greater than zero.
